# Project 2 Extra Credit Fine-Tuning Notebook

**Project:** AI vs. Human Text Detection  
**Hugging Face username:** `halaszj`

This notebook is self-contained. You do **not** need to upload JSONL files, a `data/` folder, or a requirements file.

It fine-tunes two small FLAN-T5 models and pushes them to Hugging Face:

1. `halaszj/ai-text-analyst-flan-t5`
2. `halaszj/ai-writing-coach-flan-t5`

Use **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
# Cell 1 - Confirm GPU
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not detected. Training will still run, but it will be slower. Use Runtime > Change runtime type > T4 GPU.")

In [ ]:
# Cell 2 - Install dependencies directly
# This notebook does not depend on any external requirements file.
!pip -q install "transformers==4.41.2" "datasets==2.20.0" "accelerate==0.31.0" "sentencepiece==0.2.0" "huggingface_hub==0.23.4" "evaluate==0.4.2"

In [ ]:
# Cell 3 - Imports and configuration
import os
import json
import random
import numpy as np
import torch
from datasets import Dataset
from huggingface_hub import notebook_login, whoami
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

BASE_MODEL = "google/flan-t5-small"
HF_USERNAME = "halaszj"
ANALYST_REPO_ID = f"{HF_USERNAME}/ai-text-analyst-flan-t5"
COACH_REPO_ID = f"{HF_USERNAME}/ai-writing-coach-flan-t5"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Base model:", BASE_MODEL)
print("Analyst repo:", ANALYST_REPO_ID)
print("Coach repo:", COACH_REPO_ID)

In [ ]:
# Cell 4 - Login to Hugging Face
# Paste your Hugging Face WRITE token when prompted.
# Token should start with hf_...
notebook_login()

try:
    print("Logged in as:", whoami()["name"])
except Exception as e:
    print("Login check warning:", e)

In [ ]:
# Cell 5 - Generate self-contained training datasets
# No external JSONL files are needed. The notebook creates project-specific examples here.

MODEL_NAMES = ["SVM", "Decision Tree", "AdaBoost", "FNN", "LSTM", "CNN"]
AI_TEXTS = [
    "The essay presents a balanced and comprehensive discussion using smooth transitions and consistent sentence structure throughout.",
    "This response outlines the benefits of technology in education with organized paragraphs, formal language, and predictable transitions.",
    "The passage explains the topic clearly and maintains a polished tone with minimal variation in phrasing or rhythm.",
    "The analysis is coherent, grammatically consistent, and structured in a way that presents each point evenly.",
    "The document uses highly consistent formatting, neutral wording, and a logical sequence of claims and explanations.",
]
HUMAN_TEXTS = [
    "Honestly, I had trouble understanding the assignment at first, but after reading it twice I started to see what the professor wanted.",
    "I changed my mind a few times while writing this because some parts sounded too formal and not like me.",
    "When I tested the app, I noticed one model disagreed with the rest, which made me go back and check my explanation.",
    "I think the project works better now because it does more than just give a prediction; it explains what happened.",
    "My first version was messy, but I learned that a cleaner interface makes the results easier to understand.",
]
MIXED_TEXTS = [
    "The paragraph is mostly organized, but it includes a few personal comments and uneven sentence lengths that create a mixed signal.",
    "The writing is clear and formal, although some phrases sound more natural and less automated than others.",
    "The document has strong structure, but the examples include personal wording that may reduce the AI detection confidence.",
]


def make_vote_pattern(final_label, confidence_level, disagreement=False):
    if final_label == "AI Generated":
        base_votes = ["AI", "AI", "AI", "AI", "AI", "AI"]
        if disagreement:
            base_votes[random.randrange(6)] = "Human"
    else:
        base_votes = ["Human", "Human", "Human", "Human", "Human", "Human"]
        if disagreement:
            base_votes[random.randrange(6)] = "AI"

    if confidence_level == "High":
        low, high = 88, 99
    elif confidence_level == "Medium":
        low, high = 68, 87
    else:
        low, high = 51, 67

    rows = []
    for model_name, vote in zip(MODEL_NAMES, base_votes):
        conf = random.randint(low, high)
        if disagreement and ((final_label == "AI Generated" and vote == "Human") or (final_label == "Human Written" and vote == "AI")):
            conf = random.randint(51, 62)
        rows.append((model_name, vote, conf))
    return rows


def format_input(final_label, confidence_level, votes, text, word_count, lexical_diversity, avg_sentence_length, reading_grade):
    vote_lines = "\n".join([f"- {name}: {vote} ({conf}%)" for name, vote, conf in votes])
    majority = sum(1 for _, vote, _ in votes if vote == ("AI" if final_label == "AI Generated" else "Human"))
    return f"""
Project: AI vs. Human Text Detection
Overall Prediction: {final_label}
Confidence Level: {confidence_level}
Model Consensus: {majority} of 6 models agreed
Model Votes:
{vote_lines}
Document Statistics:
- Word Count: {word_count}
- Lexical Diversity: {lexical_diversity:.2f}
- Average Sentence Length: {avg_sentence_length:.1f}
- Reading Grade Estimate: {reading_grade:.1f}
Text:
{text}
""".strip()


def analyst_output(final_label, confidence_level, votes, lexical_diversity, avg_sentence_length, text_type):
    majority = sum(1 for _, vote, _ in votes if vote == ("AI" if final_label == "AI Generated" else "Human"))
    disagreement = 6 - majority
    agreement_note = "strong" if majority >= 5 else "moderate" if majority == 4 else "mixed"
    if final_label == "AI Generated":
        evidence = "uniform sentence structure, polished transitions, consistent rhythm, and limited natural variation"
        caveat = "Any disagreement should be reviewed, but the ensemble evidence still supports an AI-generated classification."
    else:
        evidence = "personal phrasing, uneven sentence rhythm, first-person perspective, and more natural variation"
        caveat = "Some formal wording may create AI-like signals, but the overall pattern leans human-written."
    return f"""
Executive Summary: The ensemble classified the document as {final_label} with {confidence_level.lower()} confidence. Model agreement was {agreement_note}, with {majority} of 6 models supporting the final decision.

Model Agreement Analysis: The strongest evidence comes from the agreement pattern across the six classifiers. {disagreement} model(s) disagreed with the majority, which should be noted in the interpretation.

Evidence Table:
- Primary Indicators: {evidence}
- Lexical Diversity: {lexical_diversity:.2f}
- Average Sentence Length: {avg_sentence_length:.1f}
- Evidence Strength: {confidence_level}

Final Assessment: The classification is supported by both the model vote pattern and the linguistic evidence. {caveat}
""".strip()


def coach_output(final_label, confidence_level, text_type):
    if final_label == "AI Generated":
        return """
Writing Coach Summary: The text is clear and organized, but it would benefit from more natural variation and personal voice.

Priority Recommendations:
1. Vary sentence length so the rhythm feels less predictable.
2. Add specific examples or personal observations.
3. Replace repeated transitions with more natural phrasing.
4. Include a few contractions or less formal expressions where appropriate.
5. Break overly balanced paragraphs into more natural sections.

Detection Risk Table:
- Sentence Uniformity: High
- Repetition: Medium
- Personal Voice: Low
- Natural Variation: Low

Suggested Revision Strategy: Keep the meaning, but rewrite selected sentences with more uneven rhythm, concrete details, and a more personal tone.
""".strip()
    return """
Writing Coach Summary: The text already contains several human-like qualities, including personal perspective and natural sentence variation.

Priority Recommendations:
1. Keep the personal examples because they reduce AI-like tone.
2. Improve clarity without making every sentence overly polished.
3. Avoid making the structure too balanced or template-like.
4. Maintain natural transitions and informal phrasing where appropriate.
5. Use specific details to strengthen authenticity.

Detection Risk Table:
- Sentence Uniformity: Low
- Repetition: Low
- Personal Voice: High
- Natural Variation: Medium

Suggested Revision Strategy: Edit lightly for grammar and clarity while preserving the natural voice and uneven rhythm.
""".strip()

analyst_examples = []
coach_examples = []
scenarios = []

for _ in range(45):
    scenarios.append(("AI Generated", random.choice(["High", "High", "Medium"]), random.choice(AI_TEXTS), random.choice([False, False, True]), "AI"))
for _ in range(45):
    scenarios.append(("Human Written", random.choice(["High", "Medium", "Medium"]), random.choice(HUMAN_TEXTS), random.choice([False, True]), "Human"))
for _ in range(30):
    final = random.choice(["AI Generated", "Human Written"])
    scenarios.append((final, random.choice(["Medium", "Low"]), random.choice(MIXED_TEXTS), True, "Mixed"))

for final_label, confidence_level, text, disagreement, text_type in scenarios:
    votes = make_vote_pattern(final_label, confidence_level, disagreement=disagreement)
    word_count = random.randint(120, 850)
    lexical_diversity = random.uniform(0.28, 0.62) if final_label == "AI Generated" else random.uniform(0.42, 0.78)
    avg_sentence_length = random.uniform(16, 25) if final_label == "AI Generated" else random.uniform(10, 22)
    reading_grade = random.uniform(9, 14)
    inp = format_input(final_label, confidence_level, votes, text, word_count, lexical_diversity, avg_sentence_length, reading_grade)
    analyst_examples.append({"input": inp, "output": analyst_output(final_label, confidence_level, votes, lexical_diversity, avg_sentence_length, text_type)})
    coach_examples.append({"input": inp, "output": coach_output(final_label, confidence_level, text_type)})

random.shuffle(analyst_examples)
random.shuffle(coach_examples)

print("Analyst examples:", len(analyst_examples))
print("Coach examples:", len(coach_examples))
print("Sample analyst input:\n", analyst_examples[0]["input"][:500])
print("\nSample analyst output:\n", analyst_examples[0]["output"][:500])

In [ ]:
# Cell 6 - Utility functions for tokenizing, training, and testing

def build_dataset(examples, task_prefix):
    rows = []
    for ex in examples:
        rows.append({
            "source_text": f"{task_prefix}\n\n{ex['input']}",
            "target_text": ex["output"],
        })
    dataset = Dataset.from_list(rows)
    return dataset.train_test_split(test_size=0.10, seed=SEED)


def tokenize_dataset(dataset_split, tokenizer):
    def preprocess(batch):
        model_inputs = tokenizer(batch["source_text"], max_length=768, truncation=True)
        labels = tokenizer(text_target=batch["target_text"], max_length=384, truncation=True)
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs
    return dataset_split.map(preprocess, batched=True, remove_columns=["source_text", "target_text"])


def fine_tune_full_model(examples, repo_id, task_prefix, output_dir, epochs=3):
    print(f"Preparing dataset for {repo_id}")
    raw_split = build_dataset(examples, task_prefix)
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
    tokenized = tokenize_dataset(raw_split, tokenizer)
    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
    training_args = Seq2SeqTrainingArguments(
        output_dir=output_dir,
        learning_rate=5e-5,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        num_train_epochs=epochs,
        weight_decay=0.01,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        predict_with_generate=True,
        generation_max_length=384,
        logging_steps=10,
        report_to="none",
        fp16=torch.cuda.is_available(),
        push_to_hub=False,
    )
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["test"],
        data_collator=data_collator,
    )
    trainer.train()
    metrics = trainer.evaluate()
    print("Evaluation metrics:", metrics)
    print(f"Saving and pushing full fine-tuned model to {repo_id}")
    model.push_to_hub(repo_id)
    tokenizer.push_to_hub(repo_id)
    return tokenizer, model, metrics


def generate_sample(tokenizer, model, task_prefix, sample_input, max_new_tokens=220):
    prompt = f"{task_prefix}\n\n{sample_input}"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=768)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=4, do_sample=False)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

In [ ]:
# Cell 7 - Fine-tune LLM 1: AI Analyst
ANALYST_TASK_PREFIX = (
    "You are the AI Analyst for an AI vs. Human Text Detection application. "
    "Explain the ensemble model result using model votes, confidence scores, document statistics, and evidence. "
    "Return a structured explanation with an executive summary, model agreement analysis, evidence, and final assessment."
)

analyst_tokenizer, analyst_model, analyst_metrics = fine_tune_full_model(
    examples=analyst_examples,
    repo_id=ANALYST_REPO_ID,
    task_prefix=ANALYST_TASK_PREFIX,
    output_dir="./analyst_finetuned_model",
    epochs=3,
)

In [ ]:
# Cell 8 - Test LLM 1: AI Analyst
sample_analyst_input = analyst_examples[0]["input"]
print(generate_sample(analyst_tokenizer, analyst_model, ANALYST_TASK_PREFIX, sample_analyst_input))

In [ ]:
# Cell 9 - Fine-tune LLM 2: Writing Coach
COACH_TASK_PREFIX = (
    "You are the Writing Coach for an AI vs. Human Text Detection application. "
    "Use the model results and document statistics to give practical revision guidance. "
    "Return strengths, priority recommendations, detection risk, and a suggested revision strategy."
)

coach_tokenizer, coach_model, coach_metrics = fine_tune_full_model(
    examples=coach_examples,
    repo_id=COACH_REPO_ID,
    task_prefix=COACH_TASK_PREFIX,
    output_dir="./coach_finetuned_model",
    epochs=3,
)

In [ ]:
# Cell 10 - Test LLM 2: Writing Coach
sample_coach_input = coach_examples[0]["input"]
print(generate_sample(coach_tokenizer, coach_model, COACH_TASK_PREFIX, sample_coach_input))

In [ ]:
# Cell 11 - Final verification
print("Fine-tuned models should now be available at:")
print("1.", f"https://huggingface.co/{ANALYST_REPO_ID}")
print("2.", f"https://huggingface.co/{COACH_REPO_ID}")
print("\nUse these in your Hugging Face Space Variables:")
print("FINETUNED_ANALYST_MODEL =", ANALYST_REPO_ID)
print("FINETUNED_COACH_MODEL =", COACH_REPO_ID)
print("\nThen restart your Streamlit Space.")

## After the notebook finishes

1. Open your Hugging Face profile: `https://huggingface.co/halaszj`
2. Confirm these repositories exist:
   - `halaszj/ai-text-analyst-flan-t5`
   - `halaszj/ai-writing-coach-flan-t5`
3. In your Streamlit Space, add/update these **Variables**:
   - `FINETUNED_ANALYST_MODEL = halaszj/ai-text-analyst-flan-t5`
   - `FINETUNED_COACH_MODEL = halaszj/ai-writing-coach-flan-t5`
4. Add your `HF_TOKEN` as a **Secret** if your app calls private models.
5. Restart the Space.